# Feature selection for use of thrombolysis

In [1]:
# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold

## Load data, filter and create k-fold

In [2]:
features_for_selection = [
    'stroke_team',
    'age',
    'male',
    'ethnicity',
    'onset_to_arrival_time',
    'arrival_to_scan_time',
    'onset_known',
    'precise_onset_known',
    'onset_during_sleep',
    'arrive_by_ambulance',
    'infarction',
    'congestive_heart_failure',
    'hypertension',
    'atrial_fibrillation',
    'diabetes',
    'prior_stroke_tia',
    'afib_antiplatelet',
    'afib_anticoagulant',
    'any_afib_diagnosis',
    'prior_disability',
    'stroke_severity',
    'random_1',
    'random_2',
    'random_3'
]

In [3]:
all_data = pd.read_csv("../../data/sam3/cleaned_data.csv", low_memory=False)

# Add three 'random' columns to the DataFrame with random values for each row
all_data['random_1'] = np.random.rand(len(all_data))
all_data['random_2'] = np.random.binomial(1, 0.5, len(all_data))
all_data['random_3'] = np.random.randint(1, 11, len(all_data))

# List all_data columns
all_data_columns = all_data.columns.tolist()
print(all_data_columns)

# Print size of all_data
print(f"Size of all_data: {all_data.shape}")

# Remove rows with missing values in the 'thrombolysis' column
all_data = all_data[all_data['thrombolysis'].notnull()]

# Print size of all_data after removing rows with missing values in the 'thrombolysis' column
print(f"Size of all_data after removing rows with missing values in the 'thrombolysis' column: {all_data.shape}")

['stroke_team', 'age', 'male', 'ethnicity', 'onset_to_arrival_time', 'onset_known', 'precise_onset_known', 'onset_during_sleep', 'arrive_by_ambulance', 'call_to_ambulance_arrival_time', 'ambulance_on_scene_time', 'ambulance_travel_to_hospital_time', 'ambulance_wait_time_at_hospital', 'month', 'year', 'weekday', 'arrival_time_3_hour_period', 'infarction', 'lvo', 'arrival_to_scan_time', 'thrombolysis', 'thrombolysis_induced_haemorrhage', 'scan_to_thrombolysis_time', 'arrival_to_thrombolysis_time', 'onset_to_thrombolysis_time', 'thrombectomy', 'arrival_to_thrombectomy_referral_time', 'arrival_to_thrombectomy_time', 'onset_to_thrombectomy_time', 'onset_to_thrombolysis_time ', 'ai_aspects', 'aspects_score', 'perfusion_imaging_used', 'transfer_time', 'transfer_used', 'congestive_heart_failure', 'hypertension', 'atrial_fibrillation', 'diabetes', 'prior_stroke_tia', 'afib_antiplatelet', 'afib_anticoagulant', 'afib_vit_k_anticoagulant', 'afib_doac_anticoagulant', 'afib_heparin_anticoagulant', '

Filter data to teams with at least 300 admissions and 10 thrombolysis use

In [4]:
keep = []
groups = all_data.groupby('stroke_team') # creates a new object of groups of data

for index, group_df in groups: # each group has an index and a dataframe of data
    
    # Skip if total admission less than 300 or total thrombolysis < 10
    if (group_df.shape[0] < 300) or (group_df['thrombolysis'].sum() < 10):
        continue
    
    else: 
        keep.append(group_df)

# Concatenate output
data = pd.DataFrame()
data = pd.concat(keep)

n_patients_after_admission_restrictions = data.shape[0]
# Print the number of patients before and after applying the admission restrictions
print(f"Number of patients before admission restrictions: {all_data.shape[0]}")
print(f"Number of patients after admission restrictions: {n_patients_after_admission_restrictions}")
print("Difference in number of patients: ", all_data.shape[0] - n_patients_after_admission_restrictions)

Number of patients before admission restrictions: 452863
Number of patients after admission restrictions: 452115
Difference in number of patients:  748


k-fold splits

In [5]:
# Create 5-fold data stratified by stroke team and thrombolysis
strat = data['stroke_team'].map(str) + '-' + data['thrombolysis'].map(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf.get_n_splits(data, strat.values)

# Put in NumPy arrays
X = data.drop(columns=['thrombolysis']).values
y = data['thrombolysis'].values
X_col_names = list(data.drop(columns=['thrombolysis']).columns)
y_col_name = 'thrombolysis'

# Loop through the k-fold splits

train_data = []
test_data = []

counter = 0
for train_index, test_index in skf.split(X, y):  

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Create a DataFrame for train and test data
    train_df = pd.DataFrame(X_train, columns=X_col_names)
    train_df[y_col_name] = y_train
    
    test_df = pd.DataFrame(X_test, columns=X_col_names)
    test_df[y_col_name] = y_test
    
    # Append to the list
    train_data.append(train_df)
    test_data.append(test_df)

## Single feature prediction

In [6]:
# Create list to store accuracies and chosen features
roc_auc_by_feature_number = []
ci_lower_by_feature_number = []
ci_upper_by_feature_number = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through available features
for feature in available_features:

    features_to_use = [feature]  # Use only the current feature for this iteration
  
    # Set up a list to hold AUC results for this feature for each kfold
    feature_auc_kfold = []
    
    # Loop through k folds
    for k_fold in range(5):

        # Get k fold split
        train = train_data[k_fold]
        test = test_data[k_fold]

        # Get X and y
        X_train = train.drop('thrombolysis', axis=1)
        X_test = test.drop('thrombolysis', axis=1)
        y_train = train['thrombolysis']
        y_test = test['thrombolysis']

        # Restrict features
        X_train = X_train[features_to_use]
        X_test = X_test[features_to_use]

        # One hot encode hospitals if hospital in features used
        if 'stroke_team' in features_to_use:
            X_train_hosp = pd.get_dummies(
                X_train['stroke_team'], prefix = 'team')
            X_train = pd.concat([X_train, X_train_hosp], axis=1)
            X_train.drop('stroke_team', axis=1, inplace=True)
            X_test_hosp = pd.get_dummies(
                X_test['stroke_team'], prefix = 'team')
            X_test = pd.concat([X_test, X_test_hosp], axis=1)
            X_test.drop('stroke_team', axis=1, inplace=True)

        # One hot encode ethnicity if in features used
        if 'ethnicity' in features_to_use:
            X_train_eth = pd.get_dummies(
                X_train['ethnicity'], prefix = 'eth')
            X_train = pd.concat([X_train, X_train_eth], axis=1)
            X_train.drop('ethnicity', axis=1, inplace=True)
            X_test_eth = pd.get_dummies(
                X_test['ethnicity'], prefix = 'eth')
            X_test = pd.concat([X_test, X_test_eth], axis=1)
            X_test.drop('ethnicity', axis=1, inplace=True)

        # Define model
        model = XGBClassifier(verbosity = 0, seed=42)

        # Fit model
        # Ensure XGBoost-compatible dtypes (the k-fold DataFrames are object-typed)
        X_train = X_train.apply(pd.to_numeric, errors='coerce')
        X_test = X_test.apply(pd.to_numeric, errors='coerce')
        y_train = pd.to_numeric(y_train, errors='coerce')

        model.fit(X_train, y_train)

        # Get predicted y category
        y_pred = model.predict(X_test)

        # Get ROC AUC for predicted categories
        y_proba = model.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_proba[:, 1])
        feature_auc_kfold.append(roc_auc)
    
    # Get average result from all k-fold splits
    feature_auc_mean = np.mean(feature_auc_kfold)
    roc_auc_by_feature_number.append(feature_auc_mean)
    print(f"Feature: {feature}, Mean ROC AUC: {feature_auc_mean:.4f}", end=', ')

    # Get 95% confidence interval for the mean ROC AUC
    ci_lower = np.percentile(feature_auc_kfold, 2.5)
    ci_upper = np.percentile(feature_auc_kfold, 97.5)
    print(f"95% Confidence Interval for Mean ROC AUC: ({ci_lower:.4f}, {ci_upper:.4f})")

    ci_lower_by_feature_number.append(ci_lower)
    ci_upper_by_feature_number.append(ci_upper)

results = pd.DataFrame({
    'Feature': available_features,
    'Mean ROC AUC': roc_auc_by_feature_number,
    'lower_95_ci': ci_lower_by_feature_number,
    'upper_95_ci': ci_upper_by_feature_number
})

# Sort results by Mean ROC AUC in descending order
results = results.sort_values(by='Mean ROC AUC', ascending=False).reset_index(drop=True)

# Save to output folder
results.to_csv('./output/single_feature_selection_thrombolysis_results.csv', index=False)

results

Feature: stroke_team, Mean ROC AUC: 0.5695, 95% Confidence Interval for Mean ROC AUC: (0.5657, 0.5734)
Feature: age, Mean ROC AUC: 0.5383, 95% Confidence Interval for Mean ROC AUC: (0.5359, 0.5398)
Feature: male, Mean ROC AUC: 0.5114, 95% Confidence Interval for Mean ROC AUC: (0.5096, 0.5150)
Feature: ethnicity, Mean ROC AUC: 0.5056, 95% Confidence Interval for Mean ROC AUC: (0.5041, 0.5084)
Feature: onset_to_arrival_time, Mean ROC AUC: 0.8588, 95% Confidence Interval for Mean ROC AUC: (0.8568, 0.8607)
Feature: arrival_to_scan_time, Mean ROC AUC: 0.7839, 95% Confidence Interval for Mean ROC AUC: (0.7814, 0.7860)
Feature: onset_known, Mean ROC AUC: 0.6501, 95% Confidence Interval for Mean ROC AUC: (0.6495, 0.6507)
Feature: precise_onset_known, Mean ROC AUC: 0.7203, 95% Confidence Interval for Mean ROC AUC: (0.7176, 0.7221)
Feature: onset_during_sleep, Mean ROC AUC: 0.5585, 95% Confidence Interval for Mean ROC AUC: (0.5570, 0.5606)
Feature: arrive_by_ambulance, Mean ROC AUC: 0.5659, 95% 

,Feature,Mean ROC AUC,lower_95_ci,upper_95_ci
0,onset_to_arrival_time,0.858765,0.856776,0.860735
1,arrival_to_scan_time,0.783902,0.781420,0.785980
2,stroke_severity,0.722250,0.717423,0.725864
3,precise_onset_known,0.720348,0.717620,0.722081
4,onset_known,0.650138,0.649483,0.650743
5,afib_anticoagulant,0.573023,0.571193,0.574131
6,infarction,0.572778,0.571927,0.573569
7,prior_disability,0.571900,0.567216,0.574586
8,stroke_team,0.569458,0.565655,0.573433
9,arrive_by_ambulance,0.565948,0.563755,0.567146


## Feature selection

In [7]:
# Create list to store accuracies and chosen features
roc_auc_by_feature_number = []
roc_auc_by_feature_number_kfold = []
chosen_features = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through number of features
for i in range (num_features):
    
    # Reset best feature and accuracy
    best_result = 0
    best_feature = ''
    
    # Loop through available features
    for feature in available_features:

        # Create copy of already chosen features to avoid original being changed
        features_to_use = chosen_features.copy()
        # Create a list of features from features already chosen + 1 new feature
        features_to_use.append(feature)
        
        # Set up a list to hold AUC results for this feature for each kfold
        feature_auc_kfold = []
        
        # Loop through k folds
        for k_fold in range(5):

            # Get k fold split
            train = train_data[k_fold]
            test = test_data[k_fold]

            # Get X and y
            X_train = train.drop('thrombolysis', axis=1)
            X_test = test.drop('thrombolysis', axis=1)
            y_train = train['thrombolysis']
            y_test = test['thrombolysis']

            # Restrict features
            X_train = X_train[features_to_use]
            X_test = X_test[features_to_use]
    
            # One hot encode hospitals if hospital in features used
            if 'stroke_team' in features_to_use:
                X_train_hosp = pd.get_dummies(
                    X_train['stroke_team'], prefix = 'team')
                X_train = pd.concat([X_train, X_train_hosp], axis=1)
                X_train.drop('stroke_team', axis=1, inplace=True)
                X_test_hosp = pd.get_dummies(
                    X_test['stroke_team'], prefix = 'team')
                X_test = pd.concat([X_test, X_test_hosp], axis=1)
                X_test.drop('stroke_team', axis=1, inplace=True)

            # One hot encode ethnicity if in features used
            if 'ethnicity' in features_to_use:
                X_train_eth = pd.get_dummies(
                    X_train['ethnicity'], prefix = 'eth')
                X_train = pd.concat([X_train, X_train_eth], axis=1)
                X_train.drop('ethnicity', axis=1, inplace=True)
                X_test_eth = pd.get_dummies(
                    X_test['ethnicity'], prefix = 'eth')
                X_test = pd.concat([X_test, X_test_eth], axis=1)
                X_test.drop('ethnicity', axis=1, inplace=True)

            # Define model
            model = XGBClassifier(verbosity = 0, seed=42)

            # Fit model
            # Ensure XGBoost-compatible dtypes (the k-fold DataFrames are object-typed)
            X_train = X_train.apply(pd.to_numeric, errors='coerce')
            X_test = X_test.apply(pd.to_numeric, errors='coerce')
            y_train = pd.to_numeric(y_train, errors='coerce')

            model.fit(X_train, y_train)

            # Get predicted y category
            y_pred = model.predict(X_test)

            # Get ROC AUC for predicted categories
            y_proba = model.predict_proba(X_test)
            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
            feature_auc_kfold.append(roc_auc)
        
        # Get average result from all k-fold splits
        feature_auc_mean = np.mean(feature_auc_kfold)
    
        # Update chosen feature and result if this feature is a new best
        if feature_auc_mean > best_result:
            best_result = feature_auc_mean
            best_result_kfold = feature_auc_kfold
            best_feature = feature
            
    # k-fold splits are complete    
    # Add mean accuracy and AUC to record of accuracy by feature number
    roc_auc_by_feature_number.append(best_result)
    roc_auc_by_feature_number_kfold.append(best_result_kfold)
    chosen_features.append(best_feature)
    available_features.remove(best_feature)
            
    print (f'Feature {i+1:2.0f}: {best_feature}, AUC: {best_result:0.3f}')

# Save to output folder
results.to_csv('./output/feature_selection_thrombolysis_results.csv', index=False)

Feature  1: onset_to_arrival_time, AUC: 0.859
Feature  2: stroke_severity, AUC: 0.894
Feature  3: infarction, AUC: 0.925
Feature  4: afib_anticoagulant, AUC: 0.939
Feature  5: arrival_to_scan_time, AUC: 0.950
Feature  6: stroke_team, AUC: 0.955
Feature  7: prior_disability, AUC: 0.959
Feature  8: precise_onset_known, AUC: 0.962
Feature  9: onset_during_sleep, AUC: 0.963
Feature 10: prior_stroke_tia, AUC: 0.963
Feature 11: atrial_fibrillation, AUC: 0.963
Feature 12: age, AUC: 0.963
Feature 13: arrive_by_ambulance, AUC: 0.963
Feature 14: onset_known, AUC: 0.963
Feature 15: diabetes, AUC: 0.963
Feature 16: congestive_heart_failure, AUC: 0.963
Feature 17: any_afib_diagnosis, AUC: 0.963
Feature 18: afib_antiplatelet, AUC: 0.963
Feature 19: male, AUC: 0.963
Feature 20: hypertension, AUC: 0.963
Feature 21: random_3, AUC: 0.963
Feature 22: ethnicity, AUC: 0.963
Feature 23: random_2, AUC: 0.963
Feature 24: random_1, AUC: 0.963


In [8]:
# Create a DataFrame to hold the results
results_df = pd.DataFrame({
    'Feature Number': list(range(1, num_features + 1)),
    'Chosen Feature': chosen_features,
    'Mean AUC': roc_auc_by_feature_number,
})

# Save to output folder
results_df.to_csv('./output/feature_selection_thrombolysis_results.csv', index=False)

In [9]:
results_df

,Feature Number,Chosen Feature,Mean AUC
0,1,onset_to_arrival_time,0.858765
1,2,stroke_severity,0.893892
2,3,infarction,0.924673
3,4,afib_anticoagulant,0.939223
4,5,arrival_to_scan_time,0.950188
5,6,stroke_team,0.955049
6,7,prior_disability,0.959105
7,8,precise_onset_known,0.962146
8,9,onset_during_sleep,0.962661
9,10,prior_stroke_tia,0.962902
